In [2]:
# setup
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Proyek_CRM_KELOMPOK')
sys.path.append(str(PROJECT_DIR))

from project_config import FINAL_DATA_PATH, OUTPUT_DIR, REPORT_DIR

import pandas as pd

Mounted at /content/drive


In [3]:
#load data final
df = pd.read_csv(FINAL_DATA_PATH)
model_result = pd.read_csv(OUTPUT_DIR / 'model_comparison_result.csv')
risk_segment_summary = pd.read_csv(OUTPUT_DIR / 'risk_segment_summary.csv')
reason_summary = pd.read_csv(OUTPUT_DIR / 'reason_code_summary.csv')

print(df.shape)
df.head()

(286, 62)


,Age,Gender,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,latitude,longitude,Pin code,...,Good Quantity,Output,Reviews,churn_risk,risk_score,risk_segment,reason_code,crm_action,crm_channel,crm_kpi
0,20,Female,Single,Student,No Income,Post Graduate,4,12.9766,77.5993,560001,...,Moderately Important,Yes,Nil,0,0.108866,Low Risk,General Risk,"Referral reward, membership tier, dan reward r...","In-app, email, referral link","Referral rate, retention rate, review rate"
1,24,Female,Single,Student,Below Rs.10000,Graduate,3,12.9770,77.5773,560009,...,Very Important,Yes,Nil,0,0.198212,Low Risk,Service Risk,"Referral reward, membership tier, dan reward r...","In-app, email, referral link","Referral rate, retention rate, review rate"
2,22,Male,Single,Student,Below Rs.10000,Post Graduate,3,12.9551,77.6593,560017,...,Moderately Important,Yes,"Many a times payment gateways are an issue, so...",0,0.140957,Low Risk,Food Quality Risk,"Referral reward, membership tier, dan reward r...","In-app, email, referral link","Referral rate, retention rate, review rate"
3,22,Female,Single,Student,No Income,Graduate,6,12.9473,77.5616,560019,...,Important,Yes,nil,0,0.163475,Low Risk,General Risk,"Referral reward, membership tier, dan reward r...","In-app, email, referral link","Referral rate, retention rate, review rate"
4,22,Male,Single,Student,Below Rs.10000,Post Graduate,4,12.9850,77.5533,560010,...,Very Important,Yes,NIL,0,0.134960,Low Risk,Service Risk,"Referral reward, membership tier, dan reward r...","In-app, email, referral link","Referral rate, retention rate, review rate"


In [4]:
# summary final
summary = pd.DataFrame({
    'metric': [
        'total_customer',
        'actual_churn_risk_rate_percent',
        'high_risk_customer',
        'medium_risk_customer',
        'low_risk_customer',
        'best_model'
    ],
    'value': [
        len(df),
        round(df['churn_risk'].mean() * 100, 2),
        (df['risk_segment'] == 'High Risk').sum(),
        (df['risk_segment'] == 'Medium Risk').sum(),
        (df['risk_segment'] == 'Low Risk').sum(),
        'Random Forest'
    ]
})

summary

,metric,value
0,total_customer,286
1,actual_churn_risk_rate_percent,22.73
2,high_risk_customer,40
3,medium_risk_customer,33
4,low_risk_customer,213
5,best_model,Random Forest


In [5]:
#Simpan summary final
summary_path = REPORT_DIR / 'final_project_summary.csv'
summary.to_csv(summary_path, index=False)

print('Final summary disimpan:', summary_path)

Final summary disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/final_project_summary.csv


In [6]:
# Ambil Model terbaik
best_row = model_result.sort_values('f1_score', ascending=False).iloc[0]
best_row

,0
model,Random Forest
accuracy,0.931034
precision,1.0
recall,0.692308
f1_score,0.818182
roc_auc,0.955556


In [7]:
# insight final
high_risk = (df['risk_segment'] == 'High Risk').sum()
medium_risk = (df['risk_segment'] == 'Medium Risk').sum()
low_risk = (df['risk_segment'] == 'Low Risk').sum()
actual_risk_rate = df['churn_risk'].mean() * 100

narrative = f"""
Narasi Insight Final

Dataset berisi {len(df)} pelanggan food delivery. Target analisis adalah churn risk, yaitu risiko pelanggan tidak melakukan pembelian ulang. Actual churn risk rate pada dataset adalah {actual_risk_rate:.2f}%.

Model utama yang dipakai adalah Random Forest karena cocok untuk data tabular, fitur kategorikal, dan kebutuhan feature importance. Model terbaik berdasarkan tabel evaluasi adalah {best_row['model']} dengan F1-score {best_row['f1_score']:.3f} dan ROC AUC {best_row['roc_auc']:.3f}.

Hasil segmentasi membagi pelanggan menjadi:
1. High Risk: {high_risk} pelanggan
2. Medium Risk: {medium_risk} pelanggan
3. Low Risk: {low_risk} pelanggan

Strategi CRM dibuat dengan CRAM, yaitu Customer Retention Action Matrix. CRAM menghubungkan risk score, risk segment, reason code, CRM action, channel, dan KPI.

Rekomendasi utama:
1. High Risk diarahkan ke fast delivery voucher, We Miss You campaign, dan kompensasi delay.
2. Medium Risk diarahkan ke loyalty point, bundle promo, dan rekomendasi restoran.
3. Low Risk diarahkan ke referral reward, membership tier, dan reward review positif.
"""

print(narrative)


Narasi Insight Final

Dataset berisi 286 pelanggan food delivery. Target analisis adalah churn risk, yaitu risiko pelanggan tidak melakukan pembelian ulang. Actual churn risk rate pada dataset adalah 22.73%.

Model utama yang dipakai adalah Random Forest karena cocok untuk data tabular, fitur kategorikal, dan kebutuhan feature importance. Model terbaik berdasarkan tabel evaluasi adalah Random Forest dengan F1-score 0.818 dan ROC AUC 0.956.

Hasil segmentasi membagi pelanggan menjadi:
1. High Risk: 40 pelanggan
2. Medium Risk: 33 pelanggan
3. Low Risk: 213 pelanggan

Strategi CRM dibuat dengan CRAM, yaitu Customer Retention Action Matrix. CRAM menghubungkan risk score, risk segment, reason code, CRM action, channel, dan KPI.

Rekomendasi utama:
1. High Risk diarahkan ke fast delivery voucher, We Miss You campaign, dan kompensasi delay.
2. Medium Risk diarahkan ke loyalty point, bundle promo, dan rekomendasi restoran.
3. Low Risk diarahkan ke referral reward, membership tier, dan reward

In [8]:
# Simpan insight
narrative_path = REPORT_DIR / 'final_insight_narrative.txt'

with open(narrative_path, 'w') as f:
    f.write(narrative)

print('Narasi final disimpan:', narrative_path)

Narasi final disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/final_insight_narrative.txt


In [9]:
# daftar output project
output_files = []

for folder in [OUTPUT_DIR, REPORT_DIR]:
    for file in folder.glob('*'):
        output_files.append({
            'folder': str(folder),
            'file_name': file.name,
            'path': str(file)
        })

output_files_df = pd.DataFrame(output_files)
output_files_df

,folder,file_name,path
0,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,dataset_profile.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
1,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,column_dictionary_draft.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
2,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,target_distribution_raw.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
3,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,01_target_distribution_raw.png,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
4,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,missing_value_report.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
5,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,cleaning_report.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
6,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,eda_target_summary.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
7,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,eda_demographic_churn_summary.csv,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
8,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,02_churn_by_age_group.png,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...
9,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...,03_churn_by_occupation.png,/content/drive/MyDrive/Proyek_CRM_KELOMPOK/out...


In [10]:
# Simpan daftar output
output_list_path = REPORT_DIR / 'daftar_output_project.txt'

with open(output_list_path, 'w') as f:
    for item in output_files:
        f.write(item['path'] + '\n')

print('Daftar output disimpan:', output_list_path)

Daftar output disimpan: /content/drive/MyDrive/Proyek_CRM_KELOMPOK/reports/daftar_output_project.txt
